# Graph-KV structure experiments on Colab

Runs `graphkv_neurips` (Graph-KV, NeurIPS 2025) on a Colab GPU. Read `docs/neurips/COLAB.md` for the budget plan.

**Budget rules (200 compute units / month)**
1. Debug on a **T4** (or CPU) with the 1B model, 5 examples, `--metric nll`. Run the 8B model on an **L4**; you won't need an A100 this month.
2. Every run gets `--time-budget-min`. It stops cleanly; rerun the same cell to **resume** (results are on Drive).
3. Sweep with `--metric nll` (one forward pass per example), then confirm the 2-3 best settings with `--metric generate`.
4. Run the last cell (**disconnect**) when done. An idle connected runtime still burns units.

In [ ]:
#@title 1. Runtime + cost rate
# Runtime > Change runtime type: L4 for the 8B model, T4 (or CPU) for debugging.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "CPU runtime"
# Look up the real rate under the RAM/Disk meter (top right): "View resources" shows units/hour.
CU_PER_HOUR = 4.8  # conservative L4 guess; T4 about 1.2-1.8; set it to what Colab shows

In [ ]:
#@title 2. Results on Google Drive (every run resumes from here)
from google.colab import drive
drive.mount('/content/drive')
import os
RUNS = '/content/drive/MyDrive/graphkv_runs'
os.makedirs(RUNS, exist_ok=True)

In [ ]:
#@title 3. Code + dependencies (~2 min)
# Private repo: add a GitHub token with read access as a Colab secret named GH_TOKEN (key icon on the left).
import os, subprocess
BRANCH = 'graph-kv-neurIPS'
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
url = f'https://{token}@github.com/shaswata965-coder/graphkv.git' if token else 'https://github.com/shaswata965-coder/graphkv.git'
if not os.path.exists('/content/graphkv'):
    subprocess.run(['git', 'clone', '-q', '--branch', BRANCH, '--recurse-submodules', url, '/content/graphkv'], check=True)
else:
    subprocess.run(['git', '-C', '/content/graphkv', 'pull', '-q', 'origin', BRANCH], check=True)
%cd /content/graphkv
!pip -q install -e ".[neurips]"
os.environ['HF_HOME'] = '/content/hf'  # model weights go to the fast local disk (re-downloaded per session, ~3-5 min for 8B)
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
#@title 4. Unit tests (~1 min, tiny random models) - once per code update
!python -m pytest -q tests/test_gkv_model.py tests/test_gkv_graph.py tests/test_gkv_structure.py

In [ ]:
#@title 4b. Self-check on this GPU (~1 min) - every line must say ok
!python -m graphkv_neurips.gpucheck

## Debug run (T4 or CPU, ~5 min, ~0.2 units)
Use this for every code or flag change before spending L4 time. It deletes its own output so each run starts fresh.

In [ ]:
#@title 5. Smoke / debug run
DEBUG_MODEL = 'unsloth/Llama-3.2-1B-Instruct'
!rm -rf {RUNS}/debug/smoke
!python -m graphkv_neurips.run_structure --model {DEBUG_MODEL} --task 2wiki --num-examples 5 \
    --chunker fixed:128 --scorer bm25 \
    --conditions sequential,graphkv-top3,graphkv-top3:targets,noedge-top3:targets \
    --metric both --max-new-tokens 32 --warmup 2 --time-budget-min 8 --cu-per-hour {CU_PER_HOUR} \
    --output-dir {RUNS}/debug/smoke

## E0 - sanity check against the authors' outputs (L4, once)
**6a (~10 min):** with the paper's model, this implementation must reproduce the released greedy generations:
every released token should be our arg-max (teacher-forced, one forward pass per method). Rows are released
runs, columns our methods; the diagonal should be near 100%, the rest lower (the negative control).
**6b (~45 min):** free generation on 150 examples, compared with the released outputs, plus the
paper-vs-released-code position comparison (`graphkv-top3` vs `graphkv-top3@official`).

In [ ]:
#@title 6a. Reproduce the released generations (teacher-forced)
MODEL = 'ldsjmdy/Tulu3-Block-FT'
!python -m graphkv_neurips.verify_released --model {MODEL} --task 2wiki --num-examples 30 --output {RUNS}/e0_verify_2wiki.json
!python -m graphkv_neurips.verify_released --model {MODEL} --task hqa --runs block,gapemp_appr_1,gapemp_appr_3,gapemp --num-examples 30 --output {RUNS}/e0_verify_hqa.json

In [ ]:
#@title 6b. E0 replication slice (official limit of 1024 new tokens; outputs are short, so it costs little)
!python -m graphkv_neurips.run_structure --model {MODEL} --task 2wiki --num-examples 150 \
    --chunker documents --scorer retriever --max-new-tokens 1024 \
    --conditions sequential,block,graphkv-top3@official,graphkv-top3 \
    --metric generate --time-budget-min 50 --cu-per-hour {CU_PER_HOUR} --output-dir {RUNS}/e0_2wiki
!python -m graphkv_neurips.compare_official --ours {RUNS}/e0_2wiki/graphkv-top3_at_official.jsonl --task 2wiki --run gapemp_appr_3
!python -m graphkv_neurips.compare_official --ours {RUNS}/e0_2wiki/block.jsonl --task 2wiki --run block

## E3 - drop the sources after the targets are updated
| condition | query sees | question it answers |
|---|---|---|
| `graphkv-top3` | sources + updated targets | paper setting |
| `graphkv-top3:targets` | updated targets only | what survives in the targets |
| `noedge-top3:targets` | same target tokens, same positions, encoded **without** edges | control: no transfer possible |
| `noedge-top3` | sources + independent targets | edges off, nothing dropped |
| `graphkv-top3:sources` | top-3 only | how much the targets matter |

`graphkv-top3:targets` minus `noedge-top3:targets` is the information that flowed from sources into target KV.
The summary reports it with a paired bootstrap interval, overall and split by where the gold answer is
(`source_only` is the decisive slice). Stage A uses NLL on many examples; stage B generates on the key pair.

In [ ]:
#@title 7. E3 stage A (NLL, ~25 min) and stage B (generation, ~30 min)
E3 = 'graphkv-top3,graphkv-top3:targets,noedge-top3:targets,noedge-top3,graphkv-top3:sources'
!python -m graphkv_neurips.run_structure --model {MODEL} --task 2wiki --num-examples 300 \
    --chunker documents --scorer retriever --conditions {E3} --metric nll \
    --time-budget-min 35 --cu-per-hour {CU_PER_HOUR} --output-dir {RUNS}/e3_docs_nll
!python -m graphkv_neurips.run_structure --model {MODEL} --task 2wiki --num-examples 150 \
    --chunker documents --scorer retriever --conditions graphkv-top3:targets,noedge-top3:targets --metric generate \
    --time-budget-min 35 --cu-per-hour {CU_PER_HOUR} --output-dir {RUNS}/e3_docs_gen

## E1 - top-k windows instead of retrieved documents
The documents are joined into one text and cut into fixed windows; a scorer picks the top-k windows as sources.
`retriever` inherits the document score, `bm25` is lexical, `attention` lets the LLM pick (question attention over
the independently encoded windows), and `random` is the control that tells you whether relevance matters.

In [ ]:
#@title 8. E1 scorer sweep (NLL, ~15 min per scorer)
E1 = 'parallel,graphkv-top3,graphkv-top3:targets,noedge-top3:targets'
for scorer in ['retriever', 'bm25', 'attention', 'random']:
    !python -m graphkv_neurips.run_structure --model {MODEL} --task 2wiki --num-examples 300 \
        --chunker fixed:128 --scorer {scorer} --conditions {E1} --metric nll \
        --time-budget-min 20 --cu-per-hour {CU_PER_HOUR} --output-dir {RUNS}/e1_fixed128_{scorer}

## E2 - what node structure: documents, windows, sentences, paragraphs, attention shifts
Compare structures at **matched granularity** (all at ~128 tokens), then sweep granularity for the best two.
If every structure lands on the same curve when plotted against tokens per chunk, the effect is granularity, not structure.

In [ ]:
#@title 9. E2 structure at matched size, then granularity sweep (NLL, ~10 min per chunker)
E2 = 'parallel,graphkv-top3,graphkv-top3:targets'
for chunker in ['documents', 'fixed:128', 'sentences:128', 'paragraphs:128', 'attention:128', 'attention-sent:128']:
    tag = chunker.replace(':', '')
    !python -m graphkv_neurips.run_structure --model {MODEL} --task 2wiki --num-examples 200 \
        --chunker {chunker} --scorer bm25 --conditions {E2} --metric nll \
        --time-budget-min 15 --cu-per-hour {CU_PER_HOUR} --output-dir {RUNS}/e2_{tag}
# granularity sweep for one structure (repeat for the runner-up)
for size in [32, 64, 256]:
    !python -m graphkv_neurips.run_structure --model {MODEL} --task 2wiki --num-examples 200 \
        --chunker attention:{size} --scorer bm25 --conditions {E2} --metric nll \
        --time-budget-min 15 --cu-per-hour {CU_PER_HOUR} --output-dir {RUNS}/e2_attention{size}

## Paper topic classification (Table 4) - optional, ~1 h for the Cora test split
Query-only, sequential and Block-RAG (reference order shuffled with seeds 42-44) and Graph-KV on citation ego-graphs.
ArXiv-QA (`python -m graphkv_neurips.run_arxiv`) needs ~88k-264k-token contexts and an 80 GB GPU; it is not in this budget.

In [ ]:
#@title Table 4 on Cora (use --num-nodes 100 for a cheaper first look)
!python -m graphkv_neurips.run_classify --model {MODEL} --dataset cora --num-nodes 100 \
    --methods query-only,sequential,block,graphkv --time-budget-min 60 --output-dir {RUNS}/cora

In [ ]:
#@title 10. Collect every summary into one table
import glob, json, os
rows = []
for path in sorted(glob.glob(f'{RUNS}/**/summary.json', recursive=True)):
    s = json.load(open(path))
    run = os.path.relpath(os.path.dirname(path), RUNS)
    if 'conditions' not in s:  # classification summary: {method: {mean, std, runs}}
        for cond, v in s.items():
            print(f"{run:28s} {cond:28s} accuracy {v['mean']:.2f} +- {v['std']:.2f} over {v['runs']} run(s)")
        continue
    for cond, v in s['conditions'].items():
        rows.append((run, cond, v['n'], v.get('accuracy', {}).get('all'), v.get('nll'), v['n_chunks'], v['mean_chunk_tokens']))
print(f"{'run':28s} {'condition':28s} {'n':>4s} {'acc':>6s} {'nll':>7s} {'chunks':>6s} {'tok/ch':>6s}")
for r in rows:
    acc = f'{r[3]:.1f}' if r[3] is not None else '-'
    nll = f'{r[4]:.3f}' if r[4] is not None else '-'
    print(f'{r[0]:28s} {r[1]:28s} {r[2]:4d} {acc:>6s} {nll:>7s} {r[5]:6.1f} {r[6]:6.0f}')

In [ ]:
#@title 11. Stop billing: disconnect and delete the runtime
from google.colab import runtime
runtime.unassign()